# 10 | Elementos aprovados do TCC e complemento MEDINA/R4

Este caderno de empacotamento foi criado depois das analises. Exporta os pixels das sete figuras e o texto das oito tabelas do manuscrito aprovado; nao os apresenta como dados recomputados. A reproducao numerica principal esta nos cadernos 01-09. Aqui tambem se recontam os DMSO de R4 do MEDINA a partir dos escores salvos e se confere a tabela consolidada dos quatro locais.

A Figura 1 usa imagens reais de FMP/N23/R1/campo1. O arquivo de fontes registra os quatro canais, chaves dos TIFFs, hashes e janelas. Este caderno nao sintetiza imagens nem presume escala fisica. A exportacao da figura arquivada preserva exatamente a apresentacao aprovada.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil
import numpy as np
import pandas as pd
from google.colab import drive
from IPython.display import display, Image

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/mba_tcc/TCC_CellPainting_cadernos_repositorio')
REC = BASE / '04_fontes_e_referencia/recursos'
SAIDA = BASE / 'resultados_editoriais' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
SAIDA.mkdir(parents=True, exist_ok=False)
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
indice = json.loads((REC / 'manuscrito/indice_figuras_tabelas.json').read_text())
assert sum(i['tipo'] == 'figura' for i in indice) == 7
assert sum(i['tipo'] == 'tabela' for i in indice) == 8
for item in indice:
    p = REC / item['arquivo']
    if sha(p) != item['sha256']: raise RuntimeError('Recurso alterado: ' + str(p))
    shutil.copy2(p, SAIDA / p.name)
(SAIDA / 'legendas_aprovadas.json').write_text(json.dumps(indice, ensure_ascii=False, indent=2))
print('Exportados sete arquivos de figura e oito tabelas. Nao foi realizado novo treinamento.')


## 1. Conferencia numerica de MEDINA/R4

Os limites foram definidos em R1-R3. DMSO nao avaliavel nao e negativo. A lista vazia de prioridades em MEDINA torna a repeticao nao aplicavel, e nao 0%.

In [ ]:
p = REC / 'MEDINA_R4/tabelas/15_escores_reserva.csv'
scores = pd.read_csv(p)
controles = scores[scores['composto'].eq('DMSO')]
if len(scores) != 2684 or len(controles) != 196: raise RuntimeError('Coorte de R4 divergente.')
linhas = []
for bloco in ['AGP','DNA','ER','Mito','Morfologia_Cells','Morfologia_Cyto','Morfologia_Nuc']:
    v = pd.to_numeric(controles['A_' + bloco], errors='coerce').to_numpy()
    ok = np.isfinite(v)
    linhas.append({'bloco': bloco, 'n_DMSO': len(v), 'n_avaliaveis': int(ok.sum()),
                   'n_acima': int((v[ok] > 1).sum()),
                   'taxa_pct': 100 * (v[ok] > 1).mean() if ok.any() else np.nan})
recontagem = pd.DataFrame(linhas)
original = pd.read_csv(REC / 'MEDINA_R4/tabelas/15_resumo_DMSO_reserva.csv')
for coluna in ['n_DMSO','n_avaliaveis','n_acima','taxa_pct']:
    if not np.allclose(recontagem.set_index('bloco')[coluna].sort_index(),
                       original.set_index('bloco')[coluna].sort_index(), equal_nan=True):
        raise RuntimeError('Resumo divergente: ' + coluna)
recontagem.to_csv(SAIDA / 'MEDINA_R4_resumo_recontado.csv',index=False)
display(recontagem)
comparacao = pd.read_csv(REC / 'MEDINA_R4/DMSO_R4_comparacao_atualizada.csv')
comparacao.to_csv(SAIDA / 'Tabela_04_base_quantitativa.csv',index=False)
display(comparacao)
print('MEDINA: 0 candidatos no desenvolvimento; repeticao NAO APLICAVEL.')


## 2. Fontes da imagem representativa e tabela do revisor

O manifesto da Figura 1 permite localizar os TIFFs e diferenciar identificacao de canal de inferencia anatomica. A ficha humana preservada nao e recalculada nem substituida por uma interpretacao automatica.

In [ ]:
origem = pd.read_csv(REC / 'manuscrito/figura1_fontes_canais.csv')
display(origem[['site','placa','replica','poco','campo','canal','key','sha256']])
origem.to_csv(SAIDA / 'Figura_01_fontes_documentadas.csv',index=False)
fig1 = next(i for i in indice if i['tipo']=='figura' and i['numero']==1)
display(Image(filename=str(REC / fig1['arquivo'])))
print('Arquivos preservados em:', SAIDA)
